# GIAI ĐOẠN 2: TIỀN HUẤN LUYỆN BASE MODEL (PRE-TRAINING) & PROFILING KV CACHE

> **Dự án:** Xây dựng Large Language Model from Scratch (Clean Architecture)  
> **Phần cứng mục tiêu:** Kaggle GPU (T4 16GB VRAM) / Colab GPU  
> **Các tính năng bắt buộc tích hợp:**  
> - **Weights & Biases (W&B):** Tự động theo dõi train/val loss, perplexity, learning rate, VRAM và text samples.  
> - **Tự động Resume Training:** Nếu phiên bị ngắt kết nối giữa chừng, tự động nạp `last_checkpoint.pt` để train tiếp mà không phải train lại từ đầu.  
> - **Kỹ thuật tối ưu VRAM:** Mixed Precision (`torch.amp.autocast`), Gradient Accumulation, Gradient Clipping.  
> - **Đo đạc định lượng (Profiling) KV Cache:** Tốc độ (Tokens/s, TTFT, TPOT) và Bộ nhớ (Peak VRAM) khi **Bật KV Cache vs Tắt KV Cache** tại 4 mốc ngữ cảnh $L \in \{128, 512, 1024, 2048\}$ kèm bảng tổng hợp và biểu đồ trực quan.

In [ ]:
# ==============================================================================
# 1. THIẾT LẬP MÔI TRƯỜNG CLOUD & CLONE REPO BẮT BUỘC
# ==============================================================================
import os
import sys

# Trên Kaggle / Colab: clone repo nếu chưa có, hoặc cập nhật nếu đã có
if not os.path.exists('/kaggle/working/Prj_LLM') and os.path.exists('/kaggle/working'):
    %cd /kaggle/working
    !git clone -b dev https://github.com/TyTrinh/Prj_LLM.git
    %cd /kaggle/working/Prj_LLM
    !pip install -q -e .
elif os.path.exists('/kaggle/working/Prj_LLM'):
    %cd /kaggle/working/Prj_LLM
    !git pull origin dev

REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'📂 Thư mục làm việc hiện tại: {REPO_ROOT}')
!pip install -q wandb datasets matplotlib pandas
!nvidia-smi


## 1. Thiết lập Môi trường & Khởi tạo Weights & Biases (W&B)

In [ ]:
# ==============================================================================
# KẾT NỐI VỚI WEIGHTS & BIASES (W&B) QUA KAGGLE SECRETS AN TOÀN
# ==============================================================================
import os
import wandb

USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", hoặc "disabled"
WANDB_API_KEY = None

# 1. Ưu tiên đọc từ Kaggle UserSecretsClient (Add-ons -> Secrets trên Kaggle)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
    print("🔑 Đã tìm thấy WANDB_API_KEY từ Kaggle Secrets!")
except Exception:
    # 2. Dự phòng đọc từ biến môi trường hệ thống
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if USE_WANDB and WANDB_MODE == "online":
    try:
        if WANDB_API_KEY:
            wandb.login(key=WANDB_API_KEY)
        else:
            # Nếu chưa lưu secret, wandb sẽ tự động mở prompt yêu cầu nhập key an toàn (không lộ trên notebook)
            wandb.login()
        print("✅ W&B login thành công!")
    except Exception as e:
        print(f"⚠️ Không thể login W&B: {e}. Chuyển sang offline mode.")
        WANDB_MODE = "offline"


In [ ]:
import os
import sys

# 1. Kiểm tra xem thư mục repo đã có chưa, nếu chưa có (session mới) thì clone về
if not os.path.exists("/kaggle/working/Prj_LLM"):
    %cd /kaggle/working
    !git clone -b dev https://github.com/TyTrinh/Prj_LLM.git

# 2. Chuyển con trỏ làm việc vào trong Prj_LLM
%cd /kaggle/working/Prj_LLM

# 3. Thêm trực tiếp đường dẫn vào sys.path để Python luôn tìm thấy 'src' 100%
if "/kaggle/working/Prj_LLM" not in sys.path:
    sys.path.insert(0, "/kaggle/working/Prj_LLM")

print("✅ Đã kết nối thư mục mã nguồn thành công:", os.getcwd())

## 2. Chuẩn bị Dữ liệu Huấn luyện (Đồng bộ Tokenizer & Vocab Size)
Chúng ta nạp đúng từ điển Tokenizer đã huấn luyện (hoặc huấn luyện từ điển thực tế), lấy kích thước vocab thực tế gán vào cấu hình `ModelConfig` để tránh xung đột shape.

In [ ]:
import numpy as np
import os
from datasets import load_dataset
from src.tokenizer import BytePairTokenizer

tokenizer = BytePairTokenizer()
# Uu tien nap tu checkpoints/tokenizer_8k neu co, sau do den checkpoints/
vocab_file = 'checkpoints/tokenizer_8k/vocab.json' if os.path.exists('checkpoints/tokenizer_8k/vocab.json') else 'checkpoints/vocab.json'
merges_file = 'checkpoints/tokenizer_8k/merges.txt' if os.path.exists('checkpoints/tokenizer_8k/merges.txt') else 'checkpoints/merges.txt'

if os.path.exists(vocab_file) and os.path.exists(merges_file):
    print(f'Nạp từ điển Tokenizer có sẵn từ {vocab_file}...')
    tokenizer.load(vocab_file, merges_file)
else:
    print('⚠️ Không tìm thấy file tokenizer, sử dụng từ điển mặc định...')

ACTUAL_VOCAB_SIZE = len(tokenizer.vocab)
print(f'Kích thước từ điển thực tế (Vocab Size): {ACTUAL_VOCAB_SIZE:,}')

os.makedirs('data', exist_ok=True)
train_bin_path = 'data/train.bin'
val_bin_path = 'data/val.bin'

if not os.path.exists(train_bin_path):
    print('⚠️ Chưa phát hiện file data/train.bin!')
    print('👉 Gợi ý: Hãy chạy trước notebook LLM_00_train_tokenizer_and_prepare_data.ipynb để chuẩn bị dữ liệu 500K mẫu với Tokenizer 8K.')
    print('📥 Nếu muốn tokenize nhanh tạm thời (100,000 mẫu) tại đây:')
    ds = load_dataset('roneneldan/TinyStories', split='train[:100000]')
    all_tokens = []
    eos_id = tokenizer.special_tokens.get('<|eos|>', 2)
    for sample in ds:
        tokens = tokenizer.encode(sample['text']) + [eos_id]
        all_tokens.extend(tokens)
    all_tokens_np = np.array(all_tokens, dtype=np.uint16)
    split_idx = int(len(all_tokens_np) * 0.95)
    all_tokens_np[:split_idx].tofile(train_bin_path)
    all_tokens_np[split_idx:].tofile(val_bin_path)
    print(f'✅ Đã lưu {train_bin_path} ({len(all_tokens_np[:split_idx]):,} tokens)')
    print(f'✅ Đã lưu {val_bin_path} ({len(all_tokens_np[split_idx:]):,} tokens)')
else:
    print(f'✅ Đã tìm thấy dữ liệu chuẩn hóa: {train_bin_path} & {val_bin_path}.')


## 3. Khởi tạo Mô hình & Cấu hình Siêu tham số (Fit 16GB VRAM, Hỗ trợ RoPE tới 2048)

In [ ]:
from src.config import ModelConfig
from src.model import GPTModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# block_size đặt là 2048 để RoPE khởi tạo đủ dải tần số cho cả Pretrain (512) lẫn Profiling (tới 2048)
model_cfg = ModelConfig(
    vocab_size=ACTUAL_VOCAB_SIZE,
    block_size=2048,      # Hỗ trợ context window tới 2048 mà không lỗi RoPE hay Causal Mask
    n_layers=8,
    n_heads=8,
    d_model=512,
    tie_weights=True,
)

model = GPTModel(model_cfg).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"🚀 GPTModel đã khởi tạo thành công trên {device}!")
print(f"Tổng số tham số: {total_params / 1e6:.2f}M (Vừa vặn GPU 16GB, RoPE context tối đa 2048)")

## 4. Vòng lặp Huấn luyện (AMP, Grad Accumulation, Cosine LR, W&B Tracking, Auto-Resume)

> **Tính năng Auto-Resume:** Tự động khôi phục đúng `step`, `epoch`, `optimizer`, `scaler` và `best_val_loss` nếu Kaggle timeout.

In [ ]:
import os
import glob
import math
import time
import torch.nn as nn
from src.utils.checkpoint import CheckpointManager
from src.utils.tracker import WandbTracker

# Siêu tham số huấn luyện
TRAIN_BLOCK_SIZE = 512         # Ngữ cảnh huấn luyện bao phủ 100% câu chuyện
BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 4           # Effective batch size = 32 * 4 = 128 sequences
MAX_LR = 5e-4
MIN_LR = 5e-5
WARMUP_STEPS = 200
MAX_STEPS = 3000              # 3.000 steps (~98M tokens cho 500K mẫu)
SAVE_CHECKPOINT_EVERY = 500   # Lưu checkpoint định kỳ mỗi 500 steps

# Thư mục lưu checkpoint tuyệt đối an toàn
CHECKPOINT_DIR = os.path.abspath('checkpoints')
LAST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, 'base_model.pt')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Khởi tạo Optimizer & Mixed Precision Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.1, betas=(0.9, 0.95))
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# ==============================================================================
# CƠ CHẾ AUTO-RESUME THÔNG MINH (KHI BỊ DỪNG ĐỘT NGỘT HOẶC DISCONNECT SESSION)
# ==============================================================================
start_step = 0
best_val_loss = float('inf')

# 1. Tìm checkpoint gần nhất: ưu tiên last_checkpoint.pt, nếu không có thì tìm step_*.pt lớn nhất
candidate_ckpt = None
if os.path.exists(LAST_CKPT_PATH):
    candidate_ckpt = LAST_CKPT_PATH
else:
    all_step_ckpts = glob.glob(os.path.join(CHECKPOINT_DIR, 'step_*.pt'))
    if all_step_ckpts:
        # Lấy file có step cao nhất
        candidate_ckpt = max(all_step_ckpts, key=os.path.getmtime)

if candidate_ckpt and os.path.exists(candidate_ckpt):
    print(f'🔄 [PHÁT HIỆN CHECKPOINT DỞ DANG] Đang khôi phục từ: {candidate_ckpt}...')
    meta = CheckpointManager.load_checkpoint(
        candidate_ckpt,
        model=model,
        optimizer=optimizer,
        scaler=scaler if use_amp else None,
        device=device,
    )
    start_step = meta.get('step', 0)
    best_val_loss = meta.get('best_val_loss', float('inf'))
    print(f'✅ Khôi phục thành công! Sẽ huấn luyện tiếp tục từ Step {start_step + 1:,} đến Step {MAX_STEPS:,}')
    print(f'• Best Val Loss trước đó: {best_val_loss:.4f}')
else:
    print('🌱 Không tìm thấy checkpoint cũ. Bắt đầu huấn luyện mới hoàn toàn từ Step 0.')

# Khởi tạo Tracker W&B
tracker = WandbTracker(
    project_name='LLM_Stage2_Pretrain_BaseModel',
    run_name=f'Pretrain-BPE8K-3000Steps-FromStep{start_step}',
    config={
        'vocab_size': model_cfg.vocab_size,
        'd_model': model_cfg.d_model,
        'n_layers': model_cfg.n_layers,
        'n_heads': model_cfg.n_heads,
        'train_block_size': TRAIN_BLOCK_SIZE,
        'batch_size': BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'max_lr': MAX_LR,
        'warmup_steps': WARMUP_STEPS,
        'max_steps': MAX_STEPS,
    },
    enabled=USE_WANDB,
    mode=WANDB_MODE,
)

# Dataloader qua memmap
train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')

def get_batch(split='train'):
    data = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(data) - TRAIN_BLOCK_SIZE, size=BATCH_SIZE)
    x = torch.stack([torch.from_numpy(data[i : i + TRAIN_BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + TRAIN_BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

def get_lr(it):
    if it < WARMUP_STEPS:
        return MAX_LR * (it + 1) / WARMUP_STEPS
    if it > MAX_STEPS:
        return MIN_LR
    decay_ratio = (it - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

@torch.no_grad()
def estimate_loss(eval_iters=50):
    model.eval()
    losses = {}
    criterion = nn.CrossEntropyLoss()
    for split in ['train', 'val']:
        loss_sum = 0.0
        for _ in range(eval_iters):
            x, y = get_batch(split)
            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss_sum += loss.item()
        losses[split] = loss_sum / eval_iters
    model.train()
    return losses


In [ ]:
# VÒNG LẶP HUẤN LUYỆN CHÍNH (TRAINING LOOP)
criterion = nn.CrossEntropyLoss()
model.train()
t0 = time.time()

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    optimizer.zero_grad()
    accum_loss = 0.0

    # Tính trung bình loss qua toàn bộ GRAD_ACCUM_STEPS
    for micro_step in range(GRAD_ACCUM_STEPS):
        x, y = get_batch('train')
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            logits = model(x)
            micro_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            micro_loss_scaled = micro_loss / GRAD_ACCUM_STEPS

        accum_loss += micro_loss.item()
        if use_amp:
            scaler.scale(micro_loss_scaled).backward()
        else:
            micro_loss_scaled.backward()

    current_train_loss = accum_loss / GRAD_ACCUM_STEPS

    if use_amp:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # Log tiến trình và gửi metrics lên W&B mỗi 50 steps
    if (step + 1) % 50 == 0 or step == start_step:
        dt = time.time() - t0
        t0 = time.time()
        current_vram = torch.cuda.max_memory_allocated() / (1024**2) if use_amp else 0.0
        
        print(f'Step {step + 1:5d}/{MAX_STEPS} | Loss: {current_train_loss:.4f} | LR: {lr:.6f} | VRAM: {current_vram:.1f}MB | {dt:.2f}s/50steps')
        
        tracker.log({
            'train/loss': current_train_loss,
            'train/learning_rate': lr,
            'system/peak_vram_mb': current_vram,
        }, step=step + 1)

    # ==========================================================================
    # ĐÁNH GIÁ VALIDATION & LƯU CHECKPOINTS ĐỊNH KỲ (MỖI 500 STEPS)
    # ==========================================================================
    if (step + 1) % SAVE_CHECKPOINT_EVERY == 0 or (step + 1) == MAX_STEPS:
        current_step = step + 1
        losses = estimate_loss()
        val_ppl = math.exp(min(losses['val'], 20))
        print(f'\n>>> [ĐÁNH GIÁ STEP {current_step}] Train Loss = {losses["train"]:.4f}, Val Loss = {losses["val"]:.4f}, Val Perplexity = {val_ppl:.2f}')

        tracker.log({
            'val/loss': losses['val'],
            'val/perplexity': val_ppl,
        }, step=current_step)

        # 1. Cập nhật best_val_loss và lưu base_model.pt nếu đạt kỷ lục
        is_new_best = losses['val'] < best_val_loss
        if is_new_best:
            best_val_loss = losses['val']
            CheckpointManager.save_base_model(model, BEST_CKPT_PATH)
            print(f'🌟 [KỶ LỤC MỚI] Đã lưu Base Model tốt nhất tại: {BEST_CKPT_PATH}')

        # 2. Luôn lưu last_checkpoint.pt để sẵn sàng Auto-Resume bất cứ lúc nào
        CheckpointManager.save_checkpoint(
            LAST_CKPT_PATH,
            model=model,
            optimizer=optimizer,
            scaler=scaler if use_amp else None,
            step=current_step,
            best_val_loss=best_val_loss,
            stage='pretrain',
        )
        print(f'💾 [Auto-Resume] Đã cập nhật: {LAST_CKPT_PATH}')

        # 3. Lưu checkpoint riêng theo từng mốc (ví dụ: checkpoints/step_500.pt, step_1000.pt, ...)
        step_ckpt_path = os.path.join(CHECKPOINT_DIR, f'step_{current_step}.pt')
        CheckpointManager.save_checkpoint(
            step_ckpt_path,
            model=model,
            optimizer=optimizer,
            scaler=scaler if use_amp else None,
            step=current_step,
            best_val_loss=best_val_loss,
            stage='pretrain',
        )
        print(f'📦 [Periodic Snapshot] Đã lưu mốc Step {current_step} tại: {step_ckpt_path}')
        print('-' * 75)


## 5. Đo đạc Định lượng (Profiling): Bật KV Cache vs. Tắt KV Cache

Đo đạc chính xác tại 4 mốc ngữ cảnh: $L \in \{128, 512, 1024, 2048\}$.
Hàm `KVCacheProfiler.measure` tự động gọi `torch.cuda.reset_peak_memory_stats()` và `torch.cuda.synchronize()` trước mỗi lượt đo để số liệu VRAM không bị cộng dồn.

In [ ]:
from src.utils.profiler import KVCacheProfiler
import pandas as pd
import json

# Nạp trọng số mô hình tốt nhất vừa huấn luyện
if os.path.exists(BEST_CKPT_PATH):
    CheckpointManager.load_base_model(model, BEST_CKPT_PATH, device=device)
model.eval()

context_lengths = [128, 512, 1024, 2048]
benchmark_results = []

print("=" * 80)
print("📊 TIẾN HÀNH ĐO ĐẠC ĐỊNH LƯỢNG: CÓ KV CACHE VS. KHÔNG CÓ KV CACHE")
print("=" * 80)

for L in context_lengths:
    print(f"\n>>> Đang đo mốc Context Length L = {L} (Sinh 64 tokens)...")
    
    # 1. Đo khi BẬT KV Cache (reset peak stats tự động trong hàm measure)
    res_on = KVCacheProfiler.measure(model, prompt_length=L, generate_tokens=64, use_kv_cache=True, device=device)
    print(f"  [BẬT KV CACHE]  -> Tốc độ: {res_on.tokens_per_sec:6.2f} tok/s | TTFT: {res_on.ttft_ms:6.2f} ms | TPOT: {res_on.tpot_ms:6.2f} ms | VRAM: {res_on.peak_vram_mb:6.2f} MB")
    benchmark_results.append(vars(res_on))

    # 2. Đo khi TẮT KV Cache
    res_off = KVCacheProfiler.measure(model, prompt_length=L, generate_tokens=64, use_kv_cache=False, device=device)
    print(f"  [TẮT KV CACHE]  -> Tốc độ: {res_off.tokens_per_sec:6.2f} tok/s | TTFT: {res_off.ttft_ms:6.2f} ms | TPOT: {res_off.tpot_ms:6.2f} ms | VRAM: {res_off.peak_vram_mb:6.2f} MB")
    benchmark_results.append(vars(res_off))

# Lưu kết quả đo định lượng dạng JSON
os.makedirs("docs/figures", exist_ok=True)
with open("docs/figures/kv_cache_metrics.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_results, f, indent=4)

# Hiển thị bảng tổng hợp so sánh trực quan (Pandas DataFrame)
table_rows = []
for L in context_lengths:
    on = next(r for r in benchmark_results if r["context_length"] == L and r["use_kv_cache"])
    off = next(r for r in benchmark_results if r["context_length"] == L and not r["use_kv_cache"])
    speedup = on["tokens_per_sec"] / max(off["tokens_per_sec"], 1e-6)
    table_rows.append({
        "Context (L)": L,
        "KV Cache": "BẬT",
        "Speed (tok/s)": on["tokens_per_sec"],
        "TTFT (ms)": on["ttft_ms"],
        "TPOT (ms)": on["tpot_ms"],
        "Peak VRAM (MB)": on["peak_vram_mb"],
        "Speedup": f"{speedup:.2f}x",
    })
    table_rows.append({
        "Context (L)": L,
        "KV Cache": "TẮT",
        "Speed (tok/s)": off["tokens_per_sec"],
        "TTFT (ms)": off["ttft_ms"],
        "TPOT (ms)": off["tpot_ms"],
        "Peak VRAM (MB)": off["peak_vram_mb"],
        "Speedup": "1.00x",
    })

df_metrics = pd.DataFrame(table_rows)
display(df_metrics)

# Đẩy bảng metrics lên Weights & Biases
tracker.log_text_samples("benchmark/kv_cache_comparison", table_rows)

## 6. Xuất Biểu đồ So sánh Hiệu năng (Checkpoint 2 Artifact)
Vẽ 4 biểu đồ trực quan đối chiếu Tốc độ, Độ trễ (TTFT, TPOT) và Bộ nhớ (Peak VRAM) giữa Bật vs Tắt KV Cache.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metrics = [
    ("tokens_per_sec", "Tốc độ sinh (Tokens/s) - [Càng cao càng tốt]", "Tokens/giây"),
    ("ttft_ms", "First Token Latency - TTFT (ms) - [Càng thấp càng tốt]", "Mili-giây (ms)"),
    ("tpot_ms", "Next Token Latency - TPOT (ms) - [Càng thấp càng tốt]", "Mili-giây (ms)"),
    ("peak_vram_mb", "Chiếm dụng Peak VRAM (MB)", "Megabytes (MB)"),
]

for idx, (m_key, title, ylabel) in enumerate(metrics):
    ax = axes[idx]
    with_kv = [next(r[m_key] for r in benchmark_results if r["context_length"] == L and r["use_kv_cache"]) for L in context_lengths]
    without_kv = [next(r[m_key] for r in benchmark_results if r["context_length"] == L and not r["use_kv_cache"]) for L in context_lengths]

    x_indices = np.arange(len(context_lengths))
    width = 0.35

    ax.bar(x_indices - width/2, with_kv, width=width, label="Bật KV Cache", color="#2ca02c")
    ax.bar(x_indices + width/2, without_kv, width=width, label="Tắt KV Cache", color="#d62728")

    ax.set_xticks(x_indices)
    ax.set_xticklabels([f"L={L}" for L in context_lengths], fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.suptitle("KẾT QUẢ ĐO ĐẠC THỰC NGHIỆM ĐỊNH LƯỢNG HIỆU NĂNG KV CACHE", fontsize=14, fontweight="bold")
plt.tight_layout()
plot_path = "docs/figures/kv_cache_benchmark.png"
plt.savefig(plot_path, dpi=300)
plt.show()
print(f"📊 Đã xuất và lưu biểu đồ tại: {plot_path}")

# Log biểu đồ lên W&B nếu đang bật
if tracker.enabled and tracker.run:
    wandb.log({"benchmark/chart": wandb.Image(plot_path)})

## 7. Demo Sinh Văn Bản Tự Hồi Quy & Đóng W&B Session

In [ ]:
prompts = [
    "Once upon a time, there was a little girl named Lily",
    "One sunny day, a little dog found a magic ball",
    "In a small village near the mountains",
]

print("=" * 75)
print("✨ DEMO SINH VĂN BẢN VỚI BASE MODEL (KV CACHE ENABLED)")
print("=" * 75)

demo_samples = []
for p in prompts:
    input_ids = torch.tensor([tokenizer.encode(p)], device=device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=40,
        top_p=0.9,
        use_kv_cache=True,
    )
    gen_text = tokenizer.decode(output_ids[0].tolist())
    print(f"\n[Prompt]: {p}")
    print(f"[Generated]:\n{gen_text}")
    print("-" * 50)
    demo_samples.append({"prompt": p, "generated_text": gen_text})

# Log các mẫu sinh văn bản lên W&B
tracker.log_text_samples("demo/generations", demo_samples)

# Kết thúc theo dõi W&B
tracker.finish()
print("🎉 Toàn bộ quá trình Huấn luyện & Profiling đã hoàn tất thành công!")